# Naive RAG for policy PDF

This notebook builds a Naive RAG pipeline for the attached PDF:

1. Connect to Supabase, OpenRouter, and Cohere near the top.
2. Create a non-conflicting Supabase vector schema: `documents_policy_2026` and `match_documents_policy_2026`.
3. Extract text from the PDF.
4. Apply adaptive chunking: policy/notice structure-aware chunking plus recursive splitting for long sections.
5. Embed chunks locally with a multilingual sentence-transformer model and store them in Supabase.
6. Retrieve with Supabase vector search, rerank with Cohere Rerank, and answer with OpenRouter Gemma free model.

Default safety: `DRY_RUN = True`, so rows are previewed before upload. Change it to `False` after checking the chunks.


In [ ]:
# Install dependencies
%pip install -q pypdf supabase cohere openai python-dotenv pandas tqdm


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import textwrap
import time
import urllib.request

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from supabase import create_client
from tqdm.auto import tqdm

PDF_PATH = Path(r"C:\Users\vdi_kth\Desktop\AI챔피언\260702_7회차\과제\2026년_소상공인_정책자금_융자사업_공고_변경(2차).pdf")
SQL_PATH = Path(r"C:\Users\vdi_kth\Desktop\AI챔피언\260702_7회차\과제\supabase_policy_rag_schema.sql")
PROJECT_DIR = PDF_PATH.parent
ENV_PATH = PROJECT_DIR / ".env"
SUPABASE_INFO_PATHS = [
    PROJECT_DIR / "superbase.txt",
    PROJECT_DIR.parent / "\uc2e4\uc2b5" / "superbase.txt",
]

TABLE_NAME = "documents_policy_2026"
MATCH_FUNCTION = "match_documents_policy_2026"
SOURCE_NAME = "policy_fund_2026_pdf"

# Local stateless Korean-friendly character n-gram hash embedding. This produces 384-dimensional vectors.
EMBEDDING_MODEL_NAME = "local_hash_char_ngram_384"
EMBEDDING_DIMENSIONS = 384
EMBEDDING_BATCH_SIZE = 32
INSERT_BATCH_SIZE = 100

# OpenRouter free Gemma model checked on OpenRouter.
OPENROUTER_MODEL = "google/gemma-4-31b-it:free"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Cohere rerank model checked on Cohere docs.
COHERE_RERANK_MODEL = "rerank-v3.5"
USE_COHERE_RERANK = True

# Set False after chunk preview looks good and the SQL schema exists in Supabase.
DRY_RUN = True
CLEAR_EXISTING = True
VECTOR_AS_STRING = False

# Adaptive chunking controls.
MAX_CHARS = 1200
MIN_CHARS = 250
OVERLAP_CHARS = 160

load_dotenv(ENV_PATH, override=False)

print(f"PDF: {PDF_PATH}")
print(f"SQL: {SQL_PATH}")
print(f"Table: {TABLE_NAME}")
print(f"Match function: {MATCH_FUNCTION}")


In [ ]:
# Put keys here only for a temporary local run. Preferred: create .env next to this notebook.
SUPABASE_URL_DIRECT = ""
SUPABASE_KEY_DIRECT = ""
OPENROUTER_API_KEY_DIRECT = ""
COHERE_API_KEY_DIRECT = ""


def first_value(*values):
    for value in values:
        if value and str(value).strip():
            return str(value).strip()
    return None


def read_supabase_info(paths):
    for path in paths:
        if not path.exists():
            continue
        url = None
        key = None
        for line in path.read_text(encoding="utf-8").splitlines():
            value = line.strip()
            if value.startswith("https://"):
                url = value.rstrip("/")
            elif value.startswith("eyJ"):
                key = value
        if url or key:
            return url, key, path
    return None, None, None


file_url, file_key, file_path = read_supabase_info(SUPABASE_INFO_PATHS)

SUPABASE_URL = first_value(SUPABASE_URL_DIRECT, os.getenv("SUPABASE_URL"), file_url)
SUPABASE_KEY = first_value(
    SUPABASE_KEY_DIRECT,
    os.getenv("SUPABASE_SERVICE_ROLE_KEY"),
    os.getenv("SUPABASE_KEY"),
    file_key,
)
OPENROUTER_API_KEY = first_value(OPENROUTER_API_KEY_DIRECT, os.getenv("OPENROUTER_API_KEY"))
COHERE_API_KEY = first_value(COHERE_API_KEY_DIRECT, os.getenv("COHERE_API_KEY"))

missing = []
if not SUPABASE_URL:
    missing.append("SUPABASE_URL")
if not SUPABASE_KEY:
    missing.append("SUPABASE_SERVICE_ROLE_KEY or SUPABASE_KEY")
if not OPENROUTER_API_KEY:
    missing.append("OPENROUTER_API_KEY")
if USE_COHERE_RERANK and not COHERE_API_KEY:
    missing.append("COHERE_API_KEY")
if missing:
    raise RuntimeError(
        "Missing credential(s): " + ", ".join(missing) + "\n"
        "Use the DIRECT variables above or create a .env file next to this notebook."
    )

SUPABASE_URL = SUPABASE_URL.rstrip("/")
if not SUPABASE_URL.startswith("https://"):
    raise ValueError("SUPABASE_URL must start with https://")
if not OPENROUTER_API_KEY.startswith("sk-or-"):
    print("Warning: OpenRouter keys usually start with sk-or-. Continue if your key is valid.")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
openrouter_client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "Policy PDF Naive RAG Notebook",
    },
)

cohere_client = None
if USE_COHERE_RERANK:
    import cohere
    cohere_client = cohere.ClientV2(api_key=COHERE_API_KEY)

print("Connections configured. Secrets are hidden.")
print(f"Supabase URL: {SUPABASE_URL}")
print(f"Supabase info file used: {file_path if file_path else 'not used'}")
print(f"OpenRouter model: {OPENROUTER_MODEL}")
print(f"Cohere rerank: {COHERE_RERANK_MODEL if USE_COHERE_RERANK else 'disabled'}")


## Supabase SQL

Run the SQL below once in the Supabase SQL Editor. The names are intentionally unique so they do not collide with existing `documents`, `documents_test`, or `match_documents` objects.


In [ ]:
schema_sql = SQL_PATH.read_text(encoding="utf-8")
print(schema_sql)


In [ ]:
def fetch_table_schema(supabase_url: str, supabase_key: str, table_name: str):
    req = urllib.request.Request(
        f"{supabase_url.rstrip('/')}/rest/v1/",
        headers={
            "apikey": supabase_key,
            "Authorization": f"Bearer {supabase_key}",
            "Accept": "application/openapi+json",
        },
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.load(resp)
    definitions = data.get("definitions") or data.get("components", {}).get("schemas", {})
    return definitions.get(table_name)


schema = fetch_table_schema(SUPABASE_URL, SUPABASE_KEY, TABLE_NAME)
if schema is None:
    raise RuntimeError(
        f"Supabase table {TABLE_NAME!r} was not found.\n\n"
        "Do this once:\n"
        f"1. Open Supabase Dashboard > SQL Editor for this project: {SUPABASE_URL}\n"
        f"2. Copy and run the SQL from: {SQL_PATH}\n"
        "3. Make sure the final line `notify pgrst, 'reload schema';` also runs.\n"
        "4. Rerun this notebook cell.\n\n"
        "If you already ran the SQL, you may be connected to a different Supabase project/key, "
        "or Supabase's API schema cache has not refreshed yet."
    )

schema_rows = []
for column, meta in schema.get("properties", {}).items():
    schema_rows.append({"column": column, "type": meta.get("type"), "format": meta.get("format")})
display(pd.DataFrame(schema_rows))


## PDF extraction

In [ ]:
from pypdf import PdfReader


def normalize_pdf_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"-\s*\d+\s*-", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_pdf_pages(path: Path):
    reader = PdfReader(str(path))
    pages = []
    for idx, page in enumerate(reader.pages, start=1):
        text = normalize_pdf_text(page.extract_text() or "")
        if text:
            pages.append({"page": idx, "text": text})
    return pages


pages = extract_pdf_pages(PDF_PATH)
print(f"Extracted pages with text: {len(pages)}")
print(pages[0]["text"][:1200].replace("\n", " | "))


## Adaptive chunking selector

The selector measures simple document signals and routes this PDF to a strategy. For this policy announcement, the route should be structure-aware chunking plus recursive splitting for long sections.


In [ ]:
HEADING_PATTERNS = [
    re.compile(r"^\s*\d+\s+[^\n]{1,80}$"),
    re.compile(r"^\s*[\u2160-\u2169]+[.\s]+[^\n]{1,80}$"),
    re.compile(r"^\s*[\u25a1\u25a0]\s*[^\n]{1,120}"),
    re.compile(r"^\s*[\u3147\u25e6]\s*[^\n]{1,120}"),
    re.compile(r"^\s*[\u2460-\u2473]\s*[^\n]{1,120}"),
    re.compile(r"^\s*<[^>]{1,80}>\s*$"),
    re.compile(r"^\s*\[[^\]]{1,80}\]\s*$"),
]


def is_heading(line: str) -> bool:
    line = line.strip()
    if not line or len(line) > 140:
        return False
    return any(pattern.match(line) for pattern in HEADING_PATTERNS)


def analyze_document(pages):
    lines = []
    for page in pages:
        lines.extend([line.strip() for line in page["text"].splitlines() if line.strip()])
    full_text = "\n".join(line for line in lines)
    heading_count = sum(1 for line in lines if is_heading(line))
    bullet_count = len(re.findall(r"[\u25a1\u25a0\u3147\u25e6\u2460-\u2473]", full_text))
    table_tokens = [
        "\uad6c\ubd84",
        "\uacf5\uae09\uaddc\ubaa8",
        "\uc9c0\uc6d0\ub300\uc0c1",
        "\ub300\ucd9c\uae08\ub9ac",
        "\uc735\uc790\ud55c\ub3c4",
        "\ubb38\uc758\ucc98",
        "\ubcc4\ud45c",
    ]
    table_hint_count = sum(full_text.count(token) for token in table_tokens)
    avg_line_len = sum(len(line) for line in lines) / max(1, len(lines))

    if heading_count >= 8 or table_hint_count >= 8 or bullet_count >= 30:
        strategy = "policy_structural_recursive"
    elif avg_line_len < 70 and heading_count >= 3:
        strategy = "structure_recursive"
    else:
        strategy = "recursive"

    return {
        "strategy": strategy,
        "pages": len(pages),
        "chars": len(full_text),
        "lines": len(lines),
        "headings": heading_count,
        "bullets": bullet_count,
        "table_hints": table_hint_count,
        "avg_line_len": round(avg_line_len, 1),
    }


doc_profile = analyze_document(pages)
doc_profile


## Adaptive chunking

In [ ]:
def split_long_text(text: str, max_chars=MAX_CHARS, overlap=OVERLAP_CHARS):
    text = text.strip()
    if len(text) <= max_chars:
        return [text]

    units = re.split(r"(?<=\ub2e4\.)\s+|(?<=\uc694\.)\s+|(?=\n[\u25a1\u25a0\u3147\u25e6\u2460-\u2473])|\n\n+", text)
    units = [unit.strip() for unit in units if unit and unit.strip()]

    chunks = []
    current = ""
    for unit in units:
        if len(unit) > max_chars:
            if current:
                chunks.append(current.strip())
                current = ""
            start = 0
            while start < len(unit):
                end = start + max_chars
                chunks.append(unit[start:end].strip())
                start = max(0, end - overlap)
                if start >= len(unit):
                    break
            continue

        candidate = (current + "\n\n" + unit).strip() if current else unit
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            tail = current[-overlap:] if current else ""
            current = (tail + "\n\n" + unit).strip() if tail else unit

    if current:
        chunks.append(current.strip())
    return [chunk for chunk in chunks if chunk]


def page_sections(page_item):
    page_no = page_item["page"]
    lines = [line.strip() for line in page_item["text"].splitlines() if line.strip()]
    sections = []
    current_title = f"page_{page_no}"
    buffer = []

    for line in lines:
        if is_heading(line) and buffer:
            sections.append({"page_start": page_no, "page_end": page_no, "section_title": current_title, "text": "\n".join(buffer)})
            current_title = line
            buffer = [line]
        elif is_heading(line):
            current_title = line
            buffer = [line]
        else:
            buffer.append(line)

    if buffer:
        sections.append({"page_start": page_no, "page_end": page_no, "section_title": current_title, "text": "\n".join(buffer)})
    return sections


def merge_short_chunks(items, min_chars=MIN_CHARS, max_chars=MAX_CHARS):
    merged = []
    pending = None
    for item in items:
        if pending is None:
            pending = item
            continue
        if len(pending["content"]) < min_chars and len(pending["content"]) + len(item["content"]) <= max_chars:
            pending["content"] = pending["content"] + "\n\n" + item["content"]
            pending["page_end"] = item["page_end"]
            pending["section_title"] = pending["section_title"] + " | " + item["section_title"]
        else:
            merged.append(pending)
            pending = item
    if pending is not None:
        merged.append(pending)
    return merged


def adaptive_chunk_pdf(pages):
    raw_chunks = []
    strategy = doc_profile["strategy"]

    for page_item in pages:
        if strategy in {"policy_structural_recursive", "structure_recursive"}:
            sections = page_sections(page_item)
        else:
            sections = [{"page_start": page_item["page"], "page_end": page_item["page"], "section_title": f"page_{page_item['page']}", "text": page_item["text"]}]

        for section in sections:
            for part in split_long_text(section["text"]):
                raw_chunks.append({
                    "content": part,
                    "page_start": section["page_start"],
                    "page_end": section["page_end"],
                    "section_title": section["section_title"],
                })

    raw_chunks = merge_short_chunks(raw_chunks)

    final_chunks = []
    for idx, chunk in enumerate(raw_chunks, start=1):
        content = chunk["content"].strip()
        metadata = {
            "source": SOURCE_NAME,
            "source_file": PDF_PATH.name,
            "chunk_id": idx,
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "section_title": chunk["section_title"],
            "chunk_strategy": doc_profile["strategy"],
            "embedding_model": EMBEDDING_MODEL_NAME,
            "embedding_dimensions": EMBEDDING_DIMENSIONS,
            "content_sha256": hashlib.sha256(content.encode("utf-8")).hexdigest(),
        }
        final_chunks.append({"content": content, "metadata": metadata})
    return final_chunks


chunks = adaptive_chunk_pdf(pages)
chunks_path = PROJECT_DIR / "policy_pdf_adaptive_chunks.json"
chunks_path.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Adaptive chunks: {len(chunks)}")
print(f"Saved: {chunks_path}")
display(pd.DataFrame([
    {
        "chunk_id": c["metadata"]["chunk_id"],
        "pages": f"{c['metadata']['page_start']}-{c['metadata']['page_end']}",
        "section": c["metadata"]["section_title"],
        "chars": len(c["content"]),
        "preview": c["content"][:180].replace("\n", " ") + "...",
    }
    for c in chunks[:10]
]))


## Embed chunks locally

In [ ]:
def hash_embedding(text: str, dim: int = EMBEDDING_DIMENSIONS):
    vec = [0.0] * dim
    text = " ".join(str(text).lower().split())
    # Korean-friendly character n-grams. Stateless, so documents and queries use the same function.
    for n in (2, 3, 4, 5):
        if len(text) < n:
            continue
        for i in range(len(text) - n + 1):
            gram = text[i:i + n]
            h = int(hashlib.blake2b(gram.encode("utf-8"), digest_size=8).hexdigest(), 16)
            idx = h % dim
            sign = 1.0 if ((h >> 8) & 1) == 0 else -1.0
            vec[idx] += sign
    norm = sum(x * x for x in vec) ** 0.5 or 1.0
    return [round(x / norm, 8) for x in vec]


texts = [chunk["content"] for chunk in chunks]
embeddings = [hash_embedding(text) for text in tqdm(texts, desc="Embedding")]

if len(embeddings[0]) != EMBEDDING_DIMENSIONS:
    raise ValueError(f"Embedding dimension mismatch: {len(embeddings[0])} != {EMBEDDING_DIMENSIONS}")

print(f"Embeddings ready: {len(embeddings)} x {len(embeddings[0])}")


## Upload to Supabase

In [ ]:
def format_vector(vector):
    if VECTOR_AS_STRING:
        return "[" + ",".join(str(x) for x in vector) + "]"
    return vector


records = []
for chunk, embedding in zip(chunks, embeddings):
    records.append({
        "content": chunk["content"],
        "metadata": chunk["metadata"],
        "embedding": format_vector(embedding),
    })

print(f"Prepared records: {len(records)}")
print(f"Columns: {list(records[0].keys())}")

if DRY_RUN:
    print("DRY_RUN=True: upload skipped. Set DRY_RUN=False in the config cell and rerun from this cell to insert.")
else:
    if CLEAR_EXISTING:
        supabase.table(TABLE_NAME).delete().eq("metadata->>source", SOURCE_NAME).execute()
        print(f"Deleted existing rows for source={SOURCE_NAME}")

    inserted = 0
    for start in tqdm(range(0, len(records), INSERT_BATCH_SIZE), desc="Uploading"):
        batch = records[start:start + INSERT_BATCH_SIZE]
        result = supabase.table(TABLE_NAME).insert(batch).execute()
        inserted += len(result.data or batch)
    print(f"Inserted rows: {inserted}")


## Naive RAG: retrieve -> rerank -> generate

This remains a simple Naive RAG pipeline: one user question, one retrieval pass, context construction, and one generation call. Cohere rerank is inserted after vector retrieval because it was requested.


In [ ]:
def embed_query(query: str):
    return hash_embedding(query)


def retrieve_from_supabase(query: str, match_count=20):
    query_embedding = format_vector(embed_query(query))
    result = supabase.rpc(MATCH_FUNCTION, {
        "query_embedding": query_embedding,
        "match_count": match_count,
        "filter": {"source": SOURCE_NAME},
    }).execute()
    return result.data or []


def cohere_rerank(query: str, docs, top_n=5):
    if not USE_COHERE_RERANK:
        return docs[:top_n]
    if not docs:
        return []

    response = cohere_client.rerank(
        model=COHERE_RERANK_MODEL,
        query=query,
        documents=[doc["content"] for doc in docs],
        top_n=min(top_n, len(docs)),
    )

    reranked = []
    for item in response.results:
        index = item.index if hasattr(item, "index") else item["index"]
        score = item.relevance_score if hasattr(item, "relevance_score") else item["relevance_score"]
        doc = dict(docs[index])
        doc["rerank_score"] = score
        reranked.append(doc)
    return reranked


def format_context(docs):
    blocks = []
    for idx, doc in enumerate(docs, start=1):
        meta = doc.get("metadata", {})
        page = meta.get("page_start")
        section = meta.get("section_title")
        similarity = doc.get("similarity")
        rerank_score = doc.get("rerank_score")
        score_line = f"similarity={similarity:.4f}" if isinstance(similarity, (int, float)) else ""
        if isinstance(rerank_score, (int, float)):
            score_line += f", rerank={rerank_score:.4f}"
        blocks.append(
            f"[Source {idx}] page={page}, section={section}, {score_line}\n{doc['content']}"
        )
    return "\n\n".join(blocks)


def ask_openrouter(question: str, context: str):
    system_prompt = (
        "You are a Korean RAG assistant. Answer only from the provided context. "
        "If the answer is not in the context, say that the document does not provide enough evidence. "
        "Cite sources like [Source 1], [Source 2]."
    )
    user_prompt = f"Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer in Korean with concise bullet points when helpful."
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=1200,
    )
    return response.choices[0].message.content


def naive_rag_answer(question: str, vector_k=20, final_k=5):
    candidates = retrieve_from_supabase(question, match_count=vector_k)
    selected = cohere_rerank(question, candidates, top_n=final_k)
    context = format_context(selected)
    answer = ask_openrouter(question, context)
    return {"question": question, "answer": answer, "contexts": selected}


In [ ]:
# Run this after upload is complete.
RUN_SAMPLE_QUERY = False

if RUN_SAMPLE_QUERY:
    question = "2026\ub144 \uc18c\uc0c1\uacf5\uc778 \uc815\ucc45\uc790\uae08\uc758 \uc811\uc218 \uc2dc\uae30\uc640 \uc2e0\uccad \ubc29\ubc95\uc740 \ubb34\uc5c7\uc778\uac00\uc694?"
    result = naive_rag_answer(question)
    print(result["answer"])
    display(pd.DataFrame([
        {
            "page": ctx["metadata"].get("page_start"),
            "section": ctx["metadata"].get("section_title"),
            "similarity": ctx.get("similarity"),
            "rerank_score": ctx.get("rerank_score"),
            "preview": ctx["content"][:160].replace("\n", " ") + "...",
        }
        for ctx in result["contexts"]
    ]))
else:
    print("Set RUN_SAMPLE_QUERY=True after uploading to test the Naive RAG pipeline.")
